# 🎓 Machine Learning Project: Personalized AI Tutor via LLM & Reinforcement Learning
## 📌 Step 05: Bayesian Knowledge Tracing (BKT) Calibration

---

### 📖 Objective:
1. Implement the recursive Bayesian Knowledge Tracing (BKT) belief update equations:
   $$P(L_t \mid \text{Observation}) = \frac{P(L_t) \cdot P(\text{Obs} \mid L)}{P(\text{Obs})}$$
   $$P(L_{t+1}) = P(L_t \mid \text{Observation}) + (1 - P(L_t \mid \text{Observation})) \cdot P(T)$$
2. Calibrate empirical parameters ($P_{\text{init}}, P_{\text{transit}}, P_{\text{slip}}, P_{\text{guess}}$) per skill.
3. Plot simulated posterior mastery progression curves.


### Cell 1: Import Libraries & Define BKT Tracker


In [ ]:
# Cell 1: Imports & BKT Tracker Class
%matplotlib inline
import os
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("✅ Libraries Imported Successfully")

class BKTTracker:
    """Bayesian Knowledge Tracing Tracker updating mastery belief P(L_t)."""
    def __init__(self, p_init=0.25, p_transit=0.15, p_slip=0.10, p_guess=0.20):
        self.p_init = p_init
        self.p_transit = p_transit
        self.p_slip = p_slip
        self.p_guess = p_guess
        self.p_l = p_init

    def update(self, correct: int) -> float:
        if correct == 1:
            num = self.p_l * (1.0 - self.p_slip)
            den = num + (1.0 - self.p_l) * self.p_guess
        else:
            num = self.p_l * self.p_slip
            den = num + (1.0 - self.p_l) * (1.0 - self.p_guess)

        posterior = num / max(1e-7, den)
        self.p_l = float(np.clip(posterior + (1.0 - posterior) * self.p_transit, 0.01, 0.99))
        return self.p_l


### Cell 2: Calibrate Empirical Parameters on Feature Dataset


In [ ]:
# Cell 2: Calibrate BKT Parameters
cur_dir = os.path.abspath(".")
DATA_DIR = os.path.join(cur_dir, "data") if os.path.exists("data") else cur_dir
FEAT_PATH = os.path.join(DATA_DIR, "assistments_features.csv")
STATS_PATH = os.path.join(DATA_DIR, "assistments_stats.json")

df = pd.read_csv(FEAT_PATH)
bkt_params = {}

for skill in df["skill_name"].unique():
    sdf = df[df["skill_name"] == skill]
    p_init = float(np.clip(sdf.groupby("user_id").first()["correct"].mean() * 0.7 + 0.05, 0.10, 0.55))
    bkt_params[skill] = {
        "p_init": round(p_init, 4), "p_transit": 0.15, "p_slip": 0.10, "p_guess": 0.20
    }

with open(STATS_PATH, "w") as f:
    json.dump({"empirical_bkt_params": bkt_params}, f, indent=2)

print("--- Calibrated BKT Parameters Table: ---")
display(pd.DataFrame(bkt_params).T)
print(f"Saved configuration to: {STATS_PATH}")


### Cell 3: Simulate & Plot Mastery Trajectories


In [ ]:
# Cell 3: Plot Mastery Progression
t_fast = BKTTracker(p_init=0.25, p_transit=0.20, p_slip=0.08, p_guess=0.20)
t_slow = BKTTracker(p_init=0.15, p_transit=0.08, p_slip=0.15, p_guess=0.25)

fast_seq = [1, 1, 1, 0, 1, 1, 1, 1, 1, 1]
slow_seq = [0, 0, 1, 0, 1, 0, 1, 1, 1, 1]

plt.figure(figsize=(9.5, 5))
plt.plot([0.25] + [t_fast.update(a) for a in fast_seq], marker='o', lw=2.5, color='#10b981', label='Fast Learner (P_transit=0.20)')
plt.plot([0.15] + [t_slow.update(a) for a in slow_seq], marker='s', lw=2.5, color='#f59e0b', label='Struggling Learner (P_transit=0.08)')
plt.axhline(0.95, color='#6366f1', linestyle='--', label='Mastery Threshold (95%)')

plt.title('Bayesian Knowledge Tracing (BKT) - Posterior Belief Growth P(L_t)', fontsize=13, fontweight='bold')
plt.xlabel('Question Sequence Step (t)', fontsize=11)
plt.ylabel('Mastery Probability P(L_t)', fontsize=11)
plt.ylim(0, 1.05)
plt.legend()
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()
print("✅ Step 05 Completed Successfully!")
